In [1]:
%run initialize_environment.py


Environment initializing completed successfully.


# Multi-Agent Event Planner

In [2]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

llm = create_azure_llm()
tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [4]:
from langchain.agents import AgentState

class EventState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [5]:
# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 5 web searches. Count every web_search call you make.
    After 5 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [6]:
# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [7]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


### Fixing the state management issue! Using threadid w/ checkpointers for production.

In [8]:
from langgraph.checkpoint.memory import MemorySaver

# Create a checkpointer to persist state automatically
memory_saver = MemorySaver()

# Recreate coordinator with checkpointer
coordinator_with_memory = create_agent(
    model=create_azure_llm(),
    tools=[search_venues, suggest_playlist, update_state],
    state_schema=EventState,
    checkpointer=memory_saver,  # Enable automatic state persistence
    system_prompt="""
    You are a event planner & coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [9]:
def send_message_with_checkpointer(msg: str, message_history, thread_id: str):
    """
    Send a message using thread-based state management.
    State is automatically persisted and retrieved - no manual passing needed!
    """
    response = coordinator_with_memory.invoke(
        {
            "messages": message_history + [HumanMessage(content=msg)]
        },
        config={
            "configurable": {"thread_id": thread_id},
            "tags": ["WP"],
            "recursion_limit": 40
        },
    )
    
    updated_messages = response["messages"]
    print(updated_messages[-1].content)
    return updated_messages

In [10]:
thread_id = "thread_1"
msg_list_v2 = send_message_with_checkpointer("I'm from Lahore and I'd like a wedding in Gujranwala for 100 guests. Also, create a playlist with jazz genre. Make sure that playlist has multiple artists.", [], thread_id)

For your wedding in Gujranwala with 100 guests, here are some venue options:

1. Adam Palace Gujranwala
   - Capacity: 100 to 300 guests
   - Features: Central AC, prime location on GT Road, bring your own food allowed
   - Price: Affordable rates
   - Known for beautiful decoration and comfortable hall

2. Lone Palace Marriage Hall
   - Capacity: Suitable for around 100 guests
   - Features: Modern amenities, personalized event services
   - Price: Premium and elegant
   - A trusted and respected venue

3. Faletti's Grand Gujranwala
   - Features: Free bridal couple stay, free stage decorations, customizable menus, event manager
   - Price: Starting from approx 1299 PKR + tax per person
   - Known for high-quality service and elegant setup

For the wedding playlist in jazz genre, I have curated 10 tracks from multiple artists with a total duration of about 36 minutes and 17 seconds. The total cost for the playlist tracks is $9.90.

Would you like to proceed with booking one of the ven

In [11]:
# Follow-up: State (genre, destination, guest_count, origin) is automatically preserved!
msg_list_v2 = send_message_with_checkpointer("Can you please display the playlist?", msg_list_v2, thread_id)

Certainly! Here is the jazz playlist for your wedding:

1. Desafinado - Duration: 3 minutes 5 seconds
2. Garota De Ipanema - Duration: 4 minutes 45 seconds
3. Samba De Uma Nota Só (One Note Samba) - Duration: 2 minutes 17 seconds
4. Por Causa De Você - Duration: 2 minutes 50 seconds
5. Ligia - Duration: 4 minutes 12 seconds
6. Fotografia - Duration: 2 minutes 9 seconds
7. Dindi (Dindi) - Duration: 4 minutes 13 seconds
8. Se Todos Fossem Iguais A Você (Instrumental) - Duration: 2 minutes 14 seconds
9. Falando De Amor - Duration: 3 minutes 39 seconds
10. Angela - Duration: 2 minutes 49 seconds

Let me know if you'd like any changes or additional songs!


In [12]:
msg_list_v2 = send_message_with_checkpointer("Cut the list in half", msg_list_v2, thread_id)

Here is the shortened jazz playlist with the top 5 tracks for your wedding:

1. Desafinado - Duration: 3 minutes 5 seconds
2. Garota De Ipanema - Duration: 4 minutes 45 seconds
3. Samba De Uma Nota Só (One Note Samba) - Duration: 2 minutes 17 seconds
4. Por Causa De Você - Duration: 2 minutes 50 seconds
5. Ligia - Duration: 4 minutes 12 seconds

Let me know if you want to make any other adjustments!
